In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

In [2]:
# import os
# proxy = 'http://127.0.0.1:7897'
# os.environ['HTTP_PROXY'] = proxy
# os.environ['HTTPS_PROXY'] = proxy

In [3]:
import yfinance as yf

In [4]:
start_date = '2010-12-01'
end_date = '2025-12-31'
dat = yf.download('JPM', start = start_date, end = end_date, interval = '1mo')

[*********************100%***********************]  1 of 1 completed


In [5]:
dat

Price,Close,High,Low,Open,Volume
Ticker,JPM,JPM,JPM,JPM,JPM
Date,,,,,
2010-12-01,28.245085,28.711176,25.069013,25.268766,665999200
2011-01-01,29.923002,30.588846,28.398223,28.631267,827482200
2011-02-01,31.123951,32.237189,29.530757,30.130706,597586300
2011-03-01,30.730650,31.397258,28.930809,30.977297,684643600
2011-04-01,30.417341,31.863880,29.017462,31.030620,577996200
...,...,...,...,...,...
2025-08-01,298.614716,300.130475,281.594586,287.697259,148828400
2025-09-01,312.494293,315.050298,288.727577,297.465497,182562300


In [6]:
dat['r'] = dat['Close'] / dat['Close'].shift(1) - 1
dat = dat.dropna()
dat['r']

Date
2011-01-01    0.059406
2011-02-01    0.040135
2011-03-01   -0.012637
2011-04-01   -0.010195
2011-05-01   -0.047239
                ...   
2025-08-01    0.022388
2025-09-01    0.046480
2025-10-01   -0.013664
2025-11-01    0.011192
2025-12-01    0.029194
Name: r, Length: 180, dtype: float64

In [7]:
FF_all = pd.read_csv('FF_Factors_mo.csv',header=3)
FF_all

,Month,Mkt-RF,SMB,HML,RMW,CMA,RF
0,196307,-0.39,-0.48,-0.81,0.64,-1.15,0.27
1,196308,5.08,-0.80,1.70,0.40,-0.38,0.25
2,196309,-1.57,-0.43,0.00,-0.78,0.15,0.27
3,196310,2.54,-1.34,-0.04,2.79,-2.25,0.29
4,196311,-0.86,-0.85,1.73,-0.43,2.27,0.27
...,...,...,...,...,...,...,...
745,202508,1.85,4.88,4.42,-0.68,2.08,0.38
746,202509,3.39,-2.18,-1.05,-2.06,-2.22,0.33
747,202510,1.96,-1.31,-3.09,-5.21,-4.03,0.37
748,202511,-0.13,1.47,3.76,1.42,0.68,0.30


In [8]:
FF = FF_all[lambda df: df['Month']>=201101]    # FF = FF_all[FF_all['Month']>=201012]
FF

,Month,Mkt-RF,SMB,HML,RMW,CMA,RF
570,201101,1.98,-2.31,0.61,-0.64,0.81,0.01
571,201102,3.48,1.65,1.18,-1.83,0.84,0.01
572,201103,0.46,2.56,-1.84,1.73,-0.07,0.01
573,201104,2.90,-0.53,-2.44,1.12,-0.86,0.00
574,201105,-1.27,-0.72,-1.85,1.92,-1.43,0.00
...,...,...,...,...,...,...,...
745,202508,1.85,4.88,4.42,-0.68,2.08,0.38
746,202509,3.39,-2.18,-1.05,-2.06,-2.22,0.33
747,202510,1.96,-1.31,-3.09,-5.21,-4.03,0.37
748,202511,-0.13,1.47,3.76,1.42,0.68,0.30


In [9]:
FF['JPM'] = dat['r'].values * 100 - FF['RF'].values

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_15052\3315324307.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  FF['JPM'] = dat['r'].values * 100 - FF['RF'].values


In [10]:
FF = FF.set_index('Month')
FF

,Mkt-RF,SMB,HML,RMW,CMA,RF,JPM
Month,,,,,,,
201101,1.98,-2.31,0.61,-0.64,0.81,0.01,5.930565
201102,3.48,1.65,1.18,-1.83,0.84,0.01,4.003463
201103,0.46,2.56,-1.84,1.73,-0.07,0.01,-1.273660
201104,2.90,-0.53,-2.44,1.12,-0.86,0.00,-1.019532
201105,-1.27,-0.72,-1.85,1.92,-1.43,0.00,-4.723885
...,...,...,...,...,...,...,...
202508,1.85,4.88,4.42,-0.68,2.08,0.38,1.858793
202509,3.39,-2.18,-1.05,-2.06,-2.22,0.33,4.317988
202510,1.96,-1.31,-3.09,-5.21,-4.03,0.37,-1.736392


In [11]:
FF.to_csv('FFmodel_data.csv')

In [12]:
df_train = FF[lambda df: df.index<=202012] # In-Sample performance
df_test = FF[lambda df: df.index>202012] # Out-of-Sample performance

Q(1) single-factor-model

In [13]:
import statsmodels.api as sm
import matplotlib.pyplot as plt

In [14]:
Y = df_train['JPM']
X = df_train['Mkt-RF']
X

Month
201101     1.98
201102     3.48
201103     0.46
201104     2.90
201105    -1.27
          ...  
202008     7.62
202009    -3.64
202010    -2.08
202011    12.44
202012     4.63
Name: Mkt-RF, Length: 120, dtype: float64

In [15]:
X = sm.add_constant(X)
X

,const,Mkt-RF
Month,,
201101,1.0,1.98
201102,1.0,3.48
201103,1.0,0.46
201104,1.0,2.90
201105,1.0,-1.27
...,...,...
202008,1.0,7.62
202009,1.0,-3.64
202010,1.0,-2.08


In [16]:
model_s = sm.OLS(Y, X)
results_s = model_s.fit()

In [17]:
print(results_s.summary())

                            OLS Regression Results                            
Dep. Variable:                    JPM   R-squared:                       0.567
Model:                            OLS   Adj. R-squared:                  0.564
Method:                 Least Squares   F-statistic:                     154.6
Date:                Sat, 28 Feb 2026   Prob (F-statistic):           3.37e-23
Time:                        15:03:11   Log-Likelihood:                -357.78
No. Observations:                 120   AIC:                             719.6
Df Residuals:                     118   BIC:                             725.1
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1655      0.456     -0.363      0.7

In [18]:
results_s.params[0]

-0.16551601871389593

Point estimation of alpha = -0.1655, and P = 0.717 > 0.05, so we cannot reject H0: alpha = 0.
Therefore, alpha is NOT nonzero in the sampling period.

In [19]:
results_s.params[1]

1.3324960495827858

Point estimation of beta = 1.3325 > 1, and P = 0.000 < 0.01 with t = 12.435, so we can affirm that beta > 1 and JPM is a cyclical stock.

In [20]:
results_s.rsquared

0.5671693398555548

R square is 0.567, so this model explains 56.7% total variation of the single-factor model.

Q(2) three-factor-model

In [21]:
Y = df_train['JPM']
X = df_train[['Mkt-RF','SMB','HML']]
X = sm.add_constant(X)
model_ff3 = sm.OLS(Y, X)
results_ff3 = model_ff3.fit()
print(results_ff3.summary())

                            OLS Regression Results                            
Dep. Variable:                    JPM   R-squared:                       0.698
Model:                            OLS   Adj. R-squared:                  0.691
Method:                 Least Squares   F-statistic:                     89.53
Date:                Sat, 28 Feb 2026   Prob (F-statistic):           4.68e-30
Time:                        15:03:14   Log-Likelihood:                -336.11
No. Observations:                 120   AIC:                             680.2
Df Residuals:                     116   BIC:                             691.4
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.4597      0.396      1.161      0.2

In [22]:
results_ff3.params[0]

0.45973625346798314

Point estimation of alpha = 0.4597, and P = 0.248 > 0.05, so we cannot reject H0: alpha = 0. Therefore, alpha is NOT nonzero in the sampling period.

In [23]:
results_ff3.rsquared

0.6983716126246098

R square is 0.698 now compared with 0.567 estimated in the single-factor model, so the Fama-French model explains 13.1% more total variation than the single-factor model.

Q(3) five-factor-model

In [24]:
Y = df_train['JPM']
X = df_train[['Mkt-RF','SMB','HML','RMW','CMA']]
X = sm.add_constant(X)
model_ff5 = sm.OLS(Y, X)
results_ff5 = model_ff5.fit()
print(results_ff5.summary())

                            OLS Regression Results                            
Dep. Variable:                    JPM   R-squared:                       0.770
Model:                            OLS   Adj. R-squared:                  0.760
Method:                 Least Squares   F-statistic:                     76.39
Date:                Sat, 28 Feb 2026   Prob (F-statistic):           9.14e-35
Time:                        15:03:15   Log-Likelihood:                -319.80
No. Observations:                 120   AIC:                             651.6
Df Residuals:                     114   BIC:                             668.3
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.6483      0.350      1.851      0.0

In [25]:
results_ff5.params[0]

0.6483308637595689

Point estimation of alpha = 0.6483, and P = 0.067 > 0.05, so we cannot reject H0: alpha = 0. Therefore, alpha is NOT nonzero in the sampling period.

In [26]:
results_ff5.rsquared

0.770141599865072

R square is 0.770 now compared with 0.698 estimated in the three-factor model, so the five-factor model explains 0.2% more total variation than the three-factor model.

Q(4) best model based on adjusted R-square, AIC and BIC

In [29]:
from sklearn.preprocessing import StandardScaler

In [30]:
X_s = StandardScaler(with_mean=True, with_std=True).fit_transform(X)
X_s

array([[ 2.01861685e-01, -8.92025004e-01,  4.05365894e-01,
        -4.84513462e-01,  6.52630006e-01],
       [ 5.67847818e-01,  6.89537125e-01,  6.21932536e-01,
        -1.25216599e+00,  6.73410724e-01],
       [-1.69004263e-01,  1.05297691e+00, -5.25490724e-01,
         1.04434073e+00,  4.30622664e-02],
       [ 4.26333180e-01, -1.81120815e-01, -7.53455610e-01,
         6.50838177e-01, -5.04163318e-01],
       [-5.91108270e-01, -2.57003846e-01, -5.29290139e-01,
         1.16690710e+00, -8.98996967e-01],
       [-7.05783925e-01,  5.45159673e-02,  9.76132978e-02,
         1.39268726e+00, -8.92070061e-01],
       [-8.54618286e-01, -4.12763753e-01, -1.72145151e-01,
         1.65717258e+00, -1.10680415e+00],
       [-1.73542491e+00, -1.20354482e+00, -7.49656196e-01,
         2.12808548e+00, -8.85489500e-02],
       [-2.12825003e+00, -1.36329857e+00, -4.07708866e-01,
         1.21206313e+00,  1.95454201e-01],
       [ 2.48561516e+00,  1.32855213e+00,  1.96398082e-01,
        -1.52310218e+00

In [31]:
from sklearn.linear_model import LinearRegression
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

In [32]:
reg = LinearRegression().fit(X_s,Y)

In [33]:
def adjusted_r2(estimator, X, Y):
    '''
    Compute Adjusted R²
    '''
    n, p = X.shape
    Yhat = estimator.predict(X)
    RSS = np.sum((Y - Yhat)**2)
    TSS = np.sum((Y - np.mean(Y))**2)
    return 1 - RSS * (n-1) / (TSS * (n-p-1))

In [34]:
n, p = X_s.shape
sigma2 = np.sum((Y - reg.predict(X_s))**2) / (n-p-1)

In [35]:
ffs_adjR2 = SFS(LinearRegression(), k_features=(1,5), forward=True, verbose=2,
          scoring=adjusted_r2, cv=None, n_jobs=1).fit(X_s,Y)

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished

[2026-02-28 15:03:17] Features: 1/5 -- score: 0.5635012834136526[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:    0.0s finished

[2026-02-28 15:03:17] Features: 2/5 -- score: 0.6931927939222309[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    0.0s finished

[2026-02-28 15:03:17] Features: 3/5 -- score: 0.7406004186096364[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    0.0s finished

[2026-02-28 15:03:17] Features: 4/5 -- score: 0.7521247050906386[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished

[2026-02-28 15:03:17] Features: 5/5 -- score: 0.7600600910872244

In [36]:
pd.DataFrame.from_dict(ffs_adjR2.get_metric_dict()).T

C:\ProgramData\Anaconda3\lib\site-packages\numpy\core\_methods.py:262: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\ProgramData\Anaconda3\lib\site-packages\numpy\core\_methods.py:254: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


,feature_idx,cv_scores,avg_score,feature_names,ci_bound,std_dev,std_err
1,"(0,)",[0.5635012834136526],0.563501,"(0,)",NaN,0.0,NaN
2,"(0, 2)",[0.6931927939222309],0.693193,"(0, 2)",NaN,0.0,NaN
3,"(0, 2, 3)",[0.7406004186096364],0.7406,"(0, 2, 3)",NaN,0.0,NaN
4,"(0, 2, 3, 4)",[0.7521247050906386],0.752125,"(0, 2, 3, 4)",NaN,0.0,NaN
5,"(0, 1, 2, 3, 4)",[0.7600600910872244],0.76006,"(0, 1, 2, 3, 4)",NaN,0.0,NaN


Adjusted R square reaches the top 0.76 from five-factor model. So five-factor model is the best model on the training set based on adjusted R square.

In [37]:
def AIC(estimator, X, Y):
    '''
    Compute AIC
    '''
    n, p = X.shape
    return n*np.log(sigma2)+2*(p+1) 

In [38]:
def BIC(estimator, X, Y):
    '''
    Compute BIC
    '''
    n, p = X.shape
    return n*np.log(sigma2)+np.log(n)*(p+1) 

In [39]:
ffs_AIC = SFS(LinearRegression(), k_features=(1,5), forward=True, verbose=2,
          scoring=AIC, cv=None, n_jobs=1).fit(X_s,Y)

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished

[2026-02-28 15:03:18] Features: 1/5 -- score: 309.21478621504275[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:    0.0s finished

[2026-02-28 15:03:18] Features: 2/5 -- score: 311.21478621504275[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    0.0s finished

[2026-02-28 15:03:18] Features: 3/5 -- score: 313.21478621504275[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    0.0s finished

[2026-02-28 15:03:18] Features: 4/5 -- score: 315.21478621504275[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished

[2026-02-28 15:03:18] Features: 5/5 -- score: 317.21478621504275

In [40]:
pd.DataFrame.from_dict(ffs_AIC.get_metric_dict()).T

C:\ProgramData\Anaconda3\lib\site-packages\numpy\core\_methods.py:262: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\ProgramData\Anaconda3\lib\site-packages\numpy\core\_methods.py:254: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


,feature_idx,cv_scores,avg_score,feature_names,ci_bound,std_dev,std_err
1,"(0,)",[309.21478621504275],309.214786,"(0,)",NaN,0.0,NaN
2,"(0, 1)",[311.21478621504275],311.214786,"(0, 1)",NaN,0.0,NaN
3,"(0, 1, 2)",[313.21478621504275],313.214786,"(0, 1, 2)",NaN,0.0,NaN
4,"(0, 1, 2, 3)",[315.21478621504275],315.214786,"(0, 1, 2, 3)",NaN,0.0,NaN
5,"(0, 1, 2, 3, 4)",[317.21478621504275],317.214786,"(0, 1, 2, 3, 4)",NaN,0.0,NaN


In [41]:
ffs_BIC = SFS(LinearRegression(), k_features=(1,5), forward=True, verbose=2,
          scoring=BIC, cv=None, n_jobs=1).fit(X_s,Y)

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished

[2026-02-28 15:03:18] Features: 1/5 -- score: 314.7897697006068[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:    0.0s finished

[2026-02-28 15:03:18] Features: 2/5 -- score: 319.57726144338886[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    0.0s finished

[2026-02-28 15:03:18] Features: 3/5 -- score: 324.36475318617096[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    0.0s finished

[2026-02-28 15:03:18] Features: 4/5 -- score: 329.152244928953[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished

[2026-02-28 15:03:18] Features: 5/5 -- score: 333.93973667173503

In [42]:
pd.DataFrame.from_dict(ffs_BIC.get_metric_dict()).T

C:\ProgramData\Anaconda3\lib\site-packages\numpy\core\_methods.py:262: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\ProgramData\Anaconda3\lib\site-packages\numpy\core\_methods.py:254: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


,feature_idx,cv_scores,avg_score,feature_names,ci_bound,std_dev,std_err
1,"(0,)",[314.7897697006068],314.78977,"(0,)",NaN,0.0,NaN
2,"(0, 1)",[319.57726144338886],319.577261,"(0, 1)",NaN,0.0,NaN
3,"(0, 1, 2)",[324.36475318617096],324.364753,"(0, 1, 2)",NaN,0.0,NaN
4,"(0, 1, 2, 3)",[329.152244928953],329.152245,"(0, 1, 2, 3)",NaN,0.0,NaN
5,"(0, 1, 2, 3, 4)",[333.93973667173503],333.939737,"(0, 1, 2, 3, 4)",NaN,0.0,NaN


In [43]:
def AIC_BIC_ratio(estimator, X, Y):
    '''
    Compute AIC/BIC
    '''
    n, p = X.shape
    aic = n*np.log(sigma2)+2*(p+1)
    bic = n*np.log(sigma2)+np.log(n)*(p+1)
    return -aic/bic

In [44]:
ffs_AIC_BIC_ratio = SFS(LinearRegression(), k_features=(1,5), forward=True, verbose=2,
          scoring=AIC_BIC_ratio, cv=None, n_jobs=1).fit(X_s,Y)

[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.0s finished

[2026-02-28 15:03:19] Features: 1/5 -- score: -0.9822898199936219[Parallel(n_jobs=1)]: Done   4 out of   4 | elapsed:    0.0s finished

[2026-02-28 15:03:19] Features: 2/5 -- score: -0.9738326963859177[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    0.0s finished

[2026-02-28 15:03:19] Features: 3/5 -- score: -0.9656252201831294[Parallel(n_jobs=1)]: Done   2 out of   2 | elapsed:    0.0s finished

[2026-02-28 15:03:19] Features: 4/5 -- score: -0.9576564980836797[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished

[2026-02-28 15:03:19] Features: 5/5 -- score: -0.9499162614686583

In [45]:
pd.DataFrame.from_dict(ffs_AIC_BIC_ratio.get_metric_dict()).T

C:\ProgramData\Anaconda3\lib\site-packages\numpy\core\_methods.py:262: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\ProgramData\Anaconda3\lib\site-packages\numpy\core\_methods.py:254: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


,feature_idx,cv_scores,avg_score,feature_names,ci_bound,std_dev,std_err
1,"(0,)",[-0.9822898199936219],-0.98229,"(0,)",NaN,0.0,NaN
2,"(0, 1)",[-0.9738326963859177],-0.973833,"(0, 1)",NaN,0.0,NaN
3,"(0, 1, 2)",[-0.9656252201831294],-0.965625,"(0, 1, 2)",NaN,0.0,NaN
4,"(0, 1, 2, 3)",[-0.9576564980836797],-0.957656,"(0, 1, 2, 3)",NaN,0.0,NaN
5,"(0, 1, 2, 3, 4)",[-0.9499162614686583],-0.949916,"(0, 1, 2, 3, 4)",NaN,0.0,NaN


-AIC/BIC reaches the top -0.95 from five-factor model. So five-factor model is the best model on the training set based on AIC and BIC.

Q(5) prediction

In [46]:
newY = df_test['JPM']
newX = FF.loc[lambda df: (df.index>=202012) & (df.index<202512), ['Mkt-RF']] 
newX = sm.add_constant(newX)

In [49]:
Y_pred = results_s.get_prediction(newX) 

In [50]:
Y_pred.predicted_mean # excess return prediction

array([  6.00394069,  -0.25879074,   3.57879788,   4.0451715 ,
         6.45698935,   0.22090784,   3.4721982 ,   1.62002869,
         3.76534733,  -6.02849864,   8.66893279,  -2.27085978,
         4.15177118,  -8.36036672,  -3.20360701,   3.93857181,
       -12.70430385,  -0.60523972, -11.35848284,  12.58647118,
        -5.18902613, -12.61102912,  10.30790293,   6.08389045,
        -8.66684082,   8.61563295,  -3.59003087,   3.15239914,
         0.67395649,   0.27420768,   8.44240846,   4.1117963 ,
        -3.3102067 ,  -7.13447036,  -4.36287857,  11.6670489 ,
         6.31041478,   0.8072061 ,   6.59023895,   3.61877276,
        -6.36162265,   5.59086692,   3.56547292,   1.46012916,
         1.97980262,   2.13970215,  -1.49801207,   8.48238334,
        -4.36287857,   3.56547292,  -3.40348142,  -8.68016578,
        -1.2848127 ,   7.90941004,   6.31041478,   2.47282616,
         2.29960167,   4.35164559,   2.44617624,  -0.33874051])

In [51]:
R2_oss = 1 - ((np.array(newY) - Y_pred.predicted_mean)**2).sum() / ((np.array(newY) - newY.mean())**2).sum()
R2_oss # Single-factor

-0.9078421028147323

In [52]:
newY = df_test['JPM']
newX = FF.loc[lambda df: (df.index>=202012) & (df.index<202512), ['Mkt-RF','SMB','HML']] 
newX = sm.add_constant(newX)

In [53]:
Y_pred = results_ff3.get_prediction(newX) 
Y_pred.predicted_mean # excess return prediction

array([  4.35859023,   3.75526693,  11.1998231 ,  11.68529729,
         5.30657579,   7.97767917,  -4.2705605 ,   0.1688488 ,
         3.73730559,   0.42597884,   7.79333704,  -1.8292948 ,
         7.61676316,   6.11813111,   0.89177425,   2.17301982,
        -4.50144344,   8.63909212, -15.64212278,   7.69679055,
        -3.8343656 , -10.67519995,  17.96799708,   7.51844725,
        -5.79843323,   4.39403145,  -3.411788  ,  -5.87049827,
         1.04914462,  -7.0666153 ,   8.0598842 ,   8.46154537,
        -3.58375772,  -4.29842319,  -3.16148421,  12.75391024,
        11.34443348,  -1.1772813 ,   3.01488308,   8.05684784,
        -5.5896272 ,   4.2955159 ,   0.32054885,   7.75757852,
         1.20797719,  -0.30350836,   0.12545406,   8.43265668,
        -6.40903432,   5.4386755 ,   2.46874188,  -4.24627077,
        -4.02645606,   4.75704233,   4.62572945,   1.53093412,
         7.23574589,   3.40545818,  -0.36019668,   4.14746819])

In [54]:
R2_oss = 1 - ((np.array(newY) - Y_pred.predicted_mean)**2).sum() / ((np.array(newY) - newY.mean())**2).sum()
R2_oss # Three-factor

-1.1898740004917334

In [55]:
newY = df_test['JPM']
newX = FF.loc[lambda df: (df.index>=202012) & (df.index<202512), ['Mkt-RF','SMB','HML','RMW','CMA']] 
newX = sm.add_constant(newX)

In [56]:
Y_pred = results_ff5.get_prediction(newX) 
Y_pred.predicted_mean # excess return prediction

array([  4.3958364 ,   2.98584788,  13.00087364,   4.1663078 ,
         5.3530218 ,   4.83531528,  -3.2991489 ,  -4.65505663,
         5.61291944,   2.73957734,   7.72548129, -10.92231497,
        -0.05969527,   5.31125919,   1.17435463,   1.79134097,
       -10.45223886,   6.67669573, -16.17466977,   9.85219089,
         0.67757131,  -7.56716978,  10.94513548,  -0.8568152 ,
        -8.52176768,   7.70452393,  -3.61220929,  -7.16165793,
        -2.90680398,  -1.6970488 ,   5.90543176,   8.95209058,
        -4.46808055,  -4.42917202,  -3.63447434,  18.49796734,
        12.74298106,   0.31535852,   6.34861369,   7.46624705,
        -5.53121682,   2.08384219,   1.6663827 ,   6.03326664,
         0.79435426,  -0.72621915,   1.97430307,  11.02820993,
        -6.95605855,  11.76711801,   2.52827892,  -4.51212451,
         1.23600915,   0.64640216,   6.85934426,   3.24549341,
         6.15930312,   8.10019988,   8.54036341,   2.77198343])

In [57]:
R2_oss = 1 - ((np.array(newY) - Y_pred.predicted_mean)**2).sum() / ((np.array(newY) - newY.mean())**2).sum()
R2_oss # Five-factor

-1.2272188292120858

The three model OOS R Square are all negative. So, it is hard to predict the market trend. There will be more likely to overestimate with more factors.